# Imports

In [54]:
import pandas as pd
import numpy as np
from datetime import datetime
from geopy.geocoders import Nominatim
from meteostat import Point, Daily
from workalendar.america.brazil import BrazilSaoPauloCity
import holidays


# Helper Function

## Getting  raw data from ONS API

In [27]:
# We will get energy data from 2016-01-01 to 2025-01-01 (yyy-mm-dd format)

In [ ]:
years = list(range(2000, 2027)) # 2000 until 2026

In [29]:
def url_csv(year):
    year_str = str(year)
    return f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{year_str}.csv"

In [30]:
dfs = []
for year in years:
    url = url_csv(year)
    print("Downloading:", year, "AND", url)
    df = pd.read_csv(url, encoding="utf-8")
    dfs.append(df)    

Downloading: 2000 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2000.csv
Downloading: 2001 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2001.csv
Downloading: 2002 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2002.csv
Downloading: 2003 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2003.csv
Downloading: 2004 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2004.csv
Downloading: 2005 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2005.csv
Downloading: 2006 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2006.csv
Downloading: 2007 AND https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2007.csv
Downloading: 2008 AND https://ons-aws-prod-opendata.s3.amazonaws

In [31]:
raw = pd.concat(dfs, ignore_index=True)

In [32]:
#SAving raw file
raw.to_csv("Daily_Energy_Load_raw_2016_2026.csv", index=False, encoding="utf-8")

## Getting  raw data from Weather 

In [33]:
# We will get weather data from 2000-01-01 to 2025-01-01 (yyy-mm-dd format)

In [34]:
geolocator = Nominatim(user_agent="energy-Load-Prediction")
loc = geolocator.geocode("São Paulo, SP, Brazil")
lat, lon = loc.latitude, loc.longitude
lat, lon = loc.latitude, loc.longitude



In [35]:
# Classe Point retorna um ponto geográfico com latitude e longitude
sp = Point(lat, lon) # setting SP as point of weather predction
start = datetime(2000, 1, 1)
end   = datetime(2026, 2, 19)

df = Daily(sp, start, end).fetch().reset_index()

# só o que interessa
df = df.rename(columns={"time": "date", "tavg": "temp_mean_c"})
temp_df = df[["date", "temp_mean_c"]]


print("Rows:", len(temp_df))
print("Start/End Date:", temp_df["date"].min(), "---", temp_df["date"].max())
print("count of  NaN:", temp_df["temp_mean_c"].isna().mean())

temp_df.to_csv("average_temp_SP_2016_2025.csv", index=False, encoding="utf-8")


Rows: 9547
Start/End Date: 2000-01-01 00:00:00 --- 2026-02-19 00:00:00
count of  NaN: 0.01162668901225516


## Concating DFs and generating final dataset 

In [75]:
df1 = pd.read_csv("Daily_Energy_Load_raw_2016_2026.csv", sep=";", encoding="utf-8")
df2 = pd.read_csv("average_temp_SP_2016_2025.csv")

In [76]:
df1.head()

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiamwmed
0,N,Norte,2000-01-01,2243.512500
1,NE,Nordeste,2000-01-01,4646.370833
2,S,Sul,2000-01-01,4800.650000
3,SE,Sudeste/Centro-Oeste,2000-01-01,19045.995833
4,N,Norte,2000-01-02,2259.808333


In [77]:
df1 = df1[df1['id_subsistema']=='SE']
df1.head()

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiamwmed
3,SE,Sudeste/Centro-Oeste,2000-01-01,19045.995833
7,SE,Sudeste/Centro-Oeste,2000-01-02,19398.025000
11,SE,Sudeste/Centro-Oeste,2000-01-03,23061.745833
15,SE,Sudeste/Centro-Oeste,2000-01-04,24228.241667
19,SE,Sudeste/Centro-Oeste,2000-01-05,24807.937500


In [78]:
df1.drop(columns=['id_subsistema','nom_subsistema'], inplace=True)

In [79]:
df1.head()

,din_instante,val_cargaenergiamwmed
3,2000-01-01,19045.995833
7,2000-01-02,19398.025000
11,2000-01-03,23061.745833
15,2000-01-04,24228.241667
19,2000-01-05,24807.937500


In [80]:
df1.rename(columns={'din_instante': 'date', 'val_cargaenergiamwmed': 'energy_load_MWmed'}, inplace=True)

In [81]:
df1.head()

,date,energy_load_MWmed
3,2000-01-01,19045.995833
7,2000-01-02,19398.025000
11,2000-01-03,23061.745833
15,2000-01-04,24228.241667
19,2000-01-05,24807.937500


In [82]:
# Applying the business rule 
# (assumption  that the The supermarket chain’s daily consumption will be derived by assuming 
# it represents exactly 0.05% of the total verified load of the SE/CO submarket )
# since the raw data is MW median, we need to * for 24 to get the daily load
df1['energy_load_MWmed'] = df1['energy_load_MWmed'].apply(lambda x: x * 0.0005 * 24)

In [83]:
df1.head()

,date,energy_load_MWmed
3,2000-01-01,228.55195
7,2000-01-02,232.77630
11,2000-01-03,276.74095
15,2000-01-04,290.73890
19,2000-01-05,297.69525


In [84]:
df1['energy_load_MWmed'].mean()

np.float64(406.7306098024443)

In [ ]:
df1 = df1.rename(columns={"energy_load_MWmed": "consumption_mwh_day"})
# 1. pd.to_datetime: Converte strings para data (format='mixed' lida com diferentes padrões).
# 2. .dt.normalize(): "Zera" o horário (meia-noite), deixando apenas o componente da data.
df1["date"] = pd.to_datetime(df1["date"], format='mixed').dt.normalize()
df2["date"] = pd.to_datetime(df2["date"], format='mixed').dt.normalize()

In [87]:
#Merging Daily Energy and avarage temp dataframes
df_merged = df1.merge(df2, on="date", how="left") 

In [88]:
df_merged.head()

,date,consumption_mwh_day,temp_mean_c
0,2000-01-01,228.55195,22.1
1,2000-01-02,232.77630,19.8
2,2000-01-03,276.74095,20.1
3,2000-01-04,290.73890,20.4
4,2000-01-05,297.69525,21.4


In [ ]:
#df_merged.to_csv("merged.csv", index=False, encoding="utf-8")

In [90]:
# Adding the flag for holydays
df_final = df_merged.copy()
df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")# coerce :- add NAN instead of error if the date is not in the correct format
cal = BrazilSaoPauloCity()
df_final["is_holiday"] = df_final["date"].apply(lambda x: int(cal.is_holiday(x.date())) if pd.notnull(x) else np.nan)

In [91]:
df_final.head()

,date,consumption_mwh_day,temp_mean_c,is_holiday
0,2000-01-01,228.55195,22.1,1
1,2000-01-02,232.77630,19.8,0
2,2000-01-03,276.74095,20.1,0
3,2000-01-04,290.73890,20.4,0
4,2000-01-05,297.69525,21.4,0


In [92]:
df_final.to_csv("Foodshop.csv", index=False, encoding="utf-8")